TOPIC: Healthcare Cost Prediction Using Healthcare Cost Dataset

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load Healthcare Cost Dataset from Google Drive
path = "/content/drive/MyDrive/ML_TAE/healthcare_dataset.csv"
df = pd.read_csv(path)

In [ ]:
# Check dataset shape
df.shape

In [ ]:
# Display dataset information
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
df.head()

In [ ]:
# Remove unnecessary columns
df = df.drop(["Name","Doctor","Hospital","Medication"], axis=1)

In [ ]:
# Convert dates to datetime
df["Date of Admission"] = pd.to_datetime(df["Date of Admission"])
df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])

# Create new feature: Length of hospital stay
df["Length_of_Stay"] = (df["Discharge Date"] - df["Date of Admission"]).dt.days

# Drop original date columns
df = df.drop(["Date of Admission","Discharge Date"], axis=1)

In [ ]:
# Convert categorical features into numeric dummy variables
df = pd.get_dummies(df, drop_first=True, dtype=int)

In [ ]:
df.head()

In [ ]:
# Features
X = df.drop("Billing Amount", axis=1)
# Target variable
y = df["Billing Amount"]

In [ ]:
from sklearn.model_selection import train_test_split

# 🔹 70% Train / 30% Test
X_train_70, X_test_30, y_train_70, y_test_30 = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 🔹 For 70-30 split
scaler_70 = StandardScaler()
X_train_70 = scaler_70.fit_transform(X_train_70)
X_test_30 = scaler_70.transform(X_test_30)

In [ ]:
# 🔹 80% Train / 20% Test
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# 🔹 For 80-20 split
scaler_80 = StandardScaler()
X_train_80 = scaler_80.fit_transform(X_train_80)
X_test_20 = scaler_80.transform(X_test_20)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
# Define regression models
models = {
"Linear Regression": LinearRegression(),
"Decision Tree": DecisionTreeRegressor(),
"Random Forest": RandomForestRegressor(),
"Support Vector Regression": SVR(),
"Gradient Boosting": GradientBoostingRegressor(),
"KNN": KNeighborsRegressor()
}

In [ ]:
# ==============================
# STEP: TRAIN & EVALUATE (70-30)
# ==============================
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

results_70 = {}

print("\n===== 70-30 RESULTS =====")

for name, model in models.items():
    model.fit(X_train_70, y_train_70)
    y_pred = model.predict(X_test_30)

    r2 = r2_score(y_test_30, y_pred)
    mse = mean_squared_error(y_test_30, y_pred)
    mae = mean_absolute_error(y_test_30, y_pred)

    results_70[name] = [r2, mse, mae]

    print(f"{name}: R2={r2:.3f}, MSE={mse:.2f}, MAE={mae:.2f}")

In [ ]:
import pandas as pd

df_70 = pd.DataFrame(results_70, index=["R2", "MSE", "MAE"]).T

df_70.style \
    .background_gradient(cmap="Greens", subset=["R2"]) \
    .background_gradient(cmap="Reds", subset=["MSE","MAE"]) \
    .format({
        "R2": "{:.3f}",
        "MSE": "{:.2f}",
        "MAE": "{:.2f}"
    }) \
    .set_caption("📊 70-30 Model Performance")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = list(results_70.keys())

r2 = [v[0] for v in results_70.values()]
mse = [v[1] for v in results_70.values()]
mae = [v[2] for v in results_70.values()]

x = np.arange(len(models))
width = 0.25

fig, ax1 = plt.subplots()

# R2 (Left axis - Blue)
ax1.bar(x - width, r2, width, label="R2", color='blue')
ax1.set_ylabel("R2 Score")

# Second axis
ax2 = ax1.twinx()

# MSE & MAE (Right axis)
ax2.bar(x, mse, width, label="MSE", color='orange')
ax2.bar(x + width, mae, width, label="MAE", color='green')
ax2.set_ylabel("MSE / MAE")

# Labels
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=25)
ax1.set_title("70-30 Model Comparison (R2, MSE, MAE)")

# Legend combine (same as before)
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2)

plt.grid()
plt.show()

In [ ]:
best_model_70 = df_70["R2"].idxmax()
print(" Best Model (70-30):", best_model_70)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Define regression models (re-initializing as it was overwritten)
models = {
"Linear Regression": LinearRegression(),
"Decision Tree": DecisionTreeRegressor(),
"Random Forest": RandomForestRegressor(),
"Support Vector Regression": SVR(),
"Gradient Boosting": GradientBoostingRegressor(),
"KNN": KNeighborsRegressor()
}

results_80 = {}

print("===== 80-20 RESULTS ====")

for name, model in models.items():
    model.fit(X_train_80, y_train_80)
    y_pred = model.predict(X_test_20)

    r2 = r2_score(y_test_20, y_pred)
    mse = mean_squared_error(y_test_20, y_pred)
    mae = mean_absolute_error(y_test_20, y_pred)

    results_80[name] = [r2, mse, mae]

    print(f"{name}: R2={r2:.3f}, MSE={mse:.2f}, MAE={mae:.2f}")

In [ ]:
import pandas as pd

df_80 = pd.DataFrame(results_80, index=["R2", "MSE", "MAE"]).T

df_80.style \
    .background_gradient(cmap="Greens", subset=["R2"]) \
    .background_gradient(cmap="Reds", subset=["MSE","MAE"]) \
    .format({
        "R2": "{:.3f}",
        "MSE": "{:.2f}",
        "MAE": "{:.2f}"
    }) \
    .set_caption("📊 80-20 Model Performance")

In [ ]:
best_model_70 = df_70["R2"].idxmax()
print(" Best Model (70-30):", best_model_70)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_names = list(results_80.keys())

r2_scores = [values[0] for values in results_80.values()]
mse_scores = [values[1] for values in results_80.values()]
mae_scores = [values[2] for values in results_80.values()]

x = np.arange(len(models_names))
width = 0.25

plt.figure()

plt.bar(x - width, r2_scores, width, label="R2")
plt.bar(x, mse_scores, width, label="MSE")
plt.bar(x + width, mae_scores, width, label="MAE")

plt.xticks(x, models_names, rotation=25)
plt.xlabel("Models")
plt.ylabel("Values")
plt.title("80-20 Model Comparison (R2, MSE, MAE)")
plt.legend()
plt.grid()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Model names
models = list(results_80.keys())

# Extract R2 scores
r2_80 = [results_80[m][0] for m in models]
r2_70 = [results_70[m][0] for m in models]

# X positions
x = np.arange(len(models))
width = 0.35

plt.figure()

# Bars
plt.bar(x - width/2, r2_80, width, label="80-20")
plt.bar(x + width/2, r2_70, width, label="70-30")

# Labels
plt.xlabel("Models")
plt.ylabel("R2 Score")
plt.title("Model Comparison (80-20 vs 70-30)")

plt.xticks(x, models, rotation=25)
plt.legend()
plt.grid()

plt.show()